In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/flipkart_product.csv", encoding="latin1")
df.head()


,ProductName,Price,Rate,Review,Summary
0,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5,Super!,Great cooler.. excellent air flow and for this...
1,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5,Awesome,Best budget 2 fit cooler. Nice cooling
2,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3,Fair,The quality is good but the power of air is de...
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",1,Useless product,Very bad product it's a only a fan
4,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3,Fair,Ok ok product


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189874 entries, 0 to 189873
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   ProductName  189874 non-null  object
 1   Price        189873 non-null  object
 2   Rate         189873 non-null  object
 3   Review       189870 non-null  object
 4   Summary      189860 non-null  object
dtypes: object(5)
memory usage: 7.2+ MB


In [5]:
df.shape, df.columns


((189874, 5),
 Index(['ProductName', 'Price', 'Rate', 'Review', 'Summary'], dtype='object'))

In [7]:
df['Rate'].value_counts().sort_index()


Rate
1                                                               19607
2                                                                6234
3                                                               15681
4                                                               39653
5                                                              108694
Bajaj DX 2 L/W Dry Iron                                             1
Nova Plus Amaze NI 10 1100 W Dry Iron?ÿ?ÿ(Grey & Turquoise)         1
Pigeon Favourite Electric Kettle?ÿ?ÿ(1.5 L, Silver, Black)          1
s                                                                   1
Name: count, dtype: int64

In [8]:
df['Rate'].dtype


dtype('O')

In [9]:
df['Rate_clean']=pd.to_numeric(df['Rate'],errors='coerce')
df=df[df['Rate_clean'].between(1,5)]
df['Rate_clean'].value_counts().sort_index()

Rate_clean
1.0     19607
2.0      6234
3.0     15681
4.0     39653
5.0    108694
Name: count, dtype: int64

In [10]:
def map_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['Rate_clean'].apply(map_sentiment)
df['sentiment'].value_counts()


C:\Users\HP\AppData\Local\Temp\ipykernel_26808\1015103707.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['Rate_clean'].apply(map_sentiment)


sentiment
positive    148347
negative     25841
neutral      15681
Name: count, dtype: int64

In [11]:
df.columns

Index(['ProductName', 'Price', 'Rate', 'Review', 'Summary', 'Rate_clean',
       'sentiment'],
      dtype='object')

In [12]:
df.isnull().sum()

ProductName     0
Price           0
Rate            0
Review          3
Summary        13
Rate_clean      0
sentiment       0
dtype: int64

In [14]:
import re
def text_cleaning(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [15]:
df['Clean_Review'] = df['Review'].apply(text_cleaning)

C:\Users\HP\AppData\Local\Temp\ipykernel_26808\4220839399.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Clean_Review'] = df['Review'].apply(text_cleaning)


In [16]:
df['Clean_Review']

0                   super
1                 awesome
2                    fair
3         useless product
4                    fair
               ...       
189868           terrific
189869           terrific
189870           terrific
189871           just wow
189872    worth the money
Name: Clean_Review, Length: 189869, dtype: object

In [17]:
df['Review']

0                  Super!
1                 Awesome
2                    Fair
3         Useless product
4                    Fair
               ...       
189868           Terrific
189869           Terrific
189870           Terrific
189871          Just wow!
189872    Worth the money
Name: Review, Length: 189869, dtype: object

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=5000,stop_words='english')
X=tfidf.fit_transform(df['Clean_Review'])

In [23]:
df['sentiment']=df['Rate_clean'].apply(map_sentiment)
df['sentiment'].value_counts()
y=df['sentiment']

In [36]:
# Make sure all reviews are strings
df['Clean_Review'] = df['Clean_Review'].astype(str)




In [37]:
import os
# Save the CSV
df.to_csv('../data/processed/flipkart_reviews_clean.csv', index=False)


In [24]:
print(X.shape)
print(y.value_counts())

(189869, 1020)
sentiment
positive    148347
negative     25841
neutral      15681
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [26]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train,y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [27]:
y_pred=model.predict(X_test)

In [28]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.8842629167325012
              precision    recall  f1-score   support

    negative       0.59      0.96      0.73      5168
     neutral       0.74      0.86      0.79      3136
    positive       1.00      0.87      0.93     29670

    accuracy                           0.88     37974
   macro avg       0.78      0.90      0.82     37974
weighted avg       0.92      0.88      0.89     37974

